# 1. ENVIRONMENT SETUP & SPARK INITIALIZATION

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!unzip "/content/drive/MyDrive/Urban_Flow_Analytics_Dataset_csv.zip" -d "/content/dataset"

Archive:  /content/drive/MyDrive/Urban_Flow_Analytics_Dataset_csv.zip
   creating: /content/dataset/Urban_Flow_Analytics_Dataset_csv/
   creating: /content/dataset/Urban_Flow_Analytics_Dataset_csv/Urban_Flow_Analytics_Dataset_csv/
  inflating: /content/dataset/Urban_Flow_Analytics_Dataset_csv/Urban_Flow_Analytics_Dataset_csv/Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv  
  inflating: /content/dataset/Urban_Flow_Analytics_Dataset_csv/Urban_Flow_Analytics_Dataset_csv/Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv  
  inflating: /content/dataset/Urban_Flow_Analytics_Dataset_csv/Urban_Flow_Analytics_Dataset_csv/Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv  
  inflating: /content/dataset/Urban_Flow_Analytics_Dataset_csv/Urban_Flow_Analytics_Dataset_csv/Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv  
  inflating: /content/dataset/Urban_Flow_Analytics_Dataset_csv/Urban_Flow_Analytics_Dataset_csv/Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv  
  inflating: /content/dataset/Urban_Flow_Analytic

In [3]:
!apt-get update -qq > /dev/null
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!pip install -q pyspark matplotlib seaborn

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [4]:
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType
from pyspark.sql.window import Window

In [5]:
## initializing the session
spark = SparkSession.builder \
    .appName("UrbanFlowAnalytics_PySpark") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

print("Spark Session successfully created!")

Spark Session successfully created!


## Load the dataset

In [6]:
# 1. Load taxi files directly via wildcard
TAXI_DIR = "/content/dataset/Urban_Flow_Analytics_Dataset_csv/Urban_Flow_Analytics_Dataset_csv"
taxi = spark.read.csv(
    f"{TAXI_DIR}/*.csv",
    header=True,
    inferSchema=True
)

In [7]:
# 2. Load zones
zones = spark.read.csv(
    "/content/drive/MyDrive/Urban_Flow_Analytics_Zone_Dataset(in).csv",
    header=True,
    inferSchema=True
)

In [8]:
# 3. Join pickup and dropoff zone metadata
zones_pickup = zones.select(
    F.col("loc_id").alias("origin_loc_id"),
    F.col("borough_name").alias("pickup_borough"),
    F.col("zone_name").alias("pickup_zone")
)

zones_dropoff = zones.select(
    F.col("loc_id").alias("dest_loc_id"),
    F.col("borough_name").alias("dropoff_borough"),
    F.col("zone_name").alias("dropoff_zone")
)

taxi = taxi.join(zones_pickup, on="origin_loc_id", how="left") \
           .join(zones_dropoff, on="dest_loc_id", how="left")

print(f"Total Loaded Rows: {taxi.count():,}")

Total Loaded Rows: 48,601,782


# 3. PRE-IMPUTATION EXPLORATORY DATA ANALYSIS (EDA) & VISUALIZATIONS

In [10]:
#using a sample to plot safely without crashing RAM
sample_df = taxi.sample(withReplacement=False, fraction=0.01, seed=42).toPandas()

## Anomaly detection — one flag column per anomaly type

In [16]:
fare_cols = ["base_fare", "surcharge_misc", "transit_tax", "driver_tip_payment",
             "toll_total", "service_improvement_fee", "charge_total"]

### 3.1 Negative fares (any core fare/charge field below zero)

In [17]:
neg_condition = " OR ".join([f"`{c}` < 0" for c in fare_cols])
taxi = taxi.withColumn("flag_negative_fare", F.expr(neg_condition))

### 3.2 Zero distance but nonzero fare (meter ran, no distance recorded)

In [18]:
taxi = taxi.withColumn(
    "flag_zero_dist_nonzero_fare",
    (F.col("distance_miles") == 0) & (F.col("base_fare") > 0)
)

### 3.3 Zero passenger count

In [19]:
taxi = taxi.withColumn("flag_zero_riders", F.col("rider_count") == 0)

### 3.4 Dropoff before pickup

In [20]:
taxi = taxi.withColumn(
    "flag_dropoff_before_pickup",
    F.col("dropoff_timestamp") < F.col("pickup_timestamp")
)

### 3.5 Unrealistic speed (distance / duration)

In [21]:
trip_duration_hr = (
    F.col("dropoff_timestamp").cast("long") - F.col("pickup_timestamp").cast("long")
) / 3600.0

speed_mph = F.when(trip_duration_hr > 0, F.col("distance_miles") / trip_duration_hr)
taxi = taxi.withColumn("flag_unrealistic_speed", speed_mph > 80)

### 3.6 Pickup timestamp outside the dataset's actual collection window

In [22]:
taxi = taxi.withColumn(
    "flag_bad_date",
    (F.col("pickup_timestamp") < "2025-01-01") | (F.col("pickup_timestamp") > "2026-04-01")
)

### Filter anomalies

In [23]:
clean = taxi.filter(
    ~F.col("flag_negative_fare") &
    ~F.col("flag_zero_dist_nonzero_fare") &
    ~F.col("flag_zero_riders") &
    ~F.col("flag_dropoff_before_pickup") &
    ~F.col("flag_unrealistic_speed") &
    ~F.col("flag_bad_date")
)

### Imputation & Output Export

##### Conditional Imputation for rate_class_id

In [24]:
clean = clean.withColumn(
    "rate_class_id",
    F.when(F.col("rate_class_id").isNull() & (F.col("dropoff_zone") == "JFK Airport"), 2)
     .when(F.col("rate_class_id").isNull() & (F.col("dropoff_zone") == "Newark Airport"), 3)
     .otherwise(F.col("rate_class_id"))
)


##### Mode imputation (calculated dynamically)

In [25]:
rider_mode = clean.groupby("rider_count").count().orderBy(F.desc("count")).first()[0]
rate_mode = clean.groupby("rate_class_id").count().orderBy(F.desc("count")).first()[0]

# Apply Mode Imputation
clean = clean.na.fill({
    "rider_count": rider_mode,
    "rate_class_id": rate_mode
})

In [26]:
clean.write.mode("overwrite").parquet("/content/drive/MyDrive/cleaned_taxi_dataset.parquet")

## Train / validation / test split

In [27]:
window_spec = Window.orderBy("pickup_timestamp")
clean = clean.withColumn("row_id", F.row_number().over(window_spec))

total_count = clean.count()
train_cutoff = int(total_count * 0.70)
val_cutoff = int(total_count * 0.85)

train = clean.filter(F.col("row_id") <= train_cutoff).drop("row_id")
val = clean.filter((F.col("row_id") > train_cutoff) & (F.col("row_id") <= val_cutoff)).drop("row_id")
test = clean.filter(F.col("row_id") > val_cutoff).drop("row_id")

print(f"Train Rows: {train.count():,}")
print(f"Val Rows:   {val.count():,}")
print(f"Test Rows:  {test.count():,}")

# Save output in Parquet format (significantly faster and consumes much less space than CSV)
os.makedirs("/content/drive/MyDrive/cleaned_data", exist_ok=True)
train.write.mode("overwrite").parquet("/content/drive/MyDrive/cleaned_data/train.parquet")
val.write.mode("overwrite").parquet("/content/drive/MyDrive/cleaned_data/val.parquet")
test.write.mode("overwrite").parquet("/content/drive/MyDrive/cleaned_data/test.parquet")

print("Cleaned splits successfully exported to Google Drive as Parquet files!")

Train Rows: 23,937,995
Val Rows:   5,129,570
Test Rows:  5,129,571
Cleaned splits successfully exported to Google Drive as Parquet files!
